# FactChecker AI - DeBERTa Fine-Tuning

**Before running:**
1. Runtime > Change runtime type > T4 GPU
2. Add HF_TOKEN to Colab secrets (key icon, left sidebar)

**Each cell has a CHECK at the end — if it prints PASS you are good to continue.**

In [ ]:
# CELL 1 — Install
%pip install -q transformers datasets accelerate evaluate scikit-learn pandas numpy huggingface_hub sentencepiece protobuf
import transformers, datasets, torch, sklearn, evaluate as ev
print(f'transformers: {transformers.__version__}')
print(f'torch:        {torch.__version__}')
print(f'datasets:     {datasets.__version__}')
print(f'sklearn:      {sklearn.__version__}')
print('CHECK: PASS - all packages installed')

In [ ]:
# CELL 2 — Auth + GPU
import torch
from huggingface_hub import login

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    login(token=HF_TOKEN)
    print('HuggingFace: LOGGED IN')
except Exception as e:
    print(f'HuggingFace: login failed ({e}) - model will save locally only')
    HF_TOKEN = None

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cuda':
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory // 1024**3
    print(f'GPU: {gpu} ({mem} GB)')
    print('CHECK: PASS - GPU ready')
else:
    print('CHECK: WARN - No GPU! Go to Runtime > Change runtime type > T4 GPU')

HF_USERNAME = 'Bharat2004'
MODEL_NAME  = f'{HF_USERNAME}/factchecker-deberta'
print(f'Will push to: {MODEL_NAME}')

In [ ]:
# CELL 3 — Load datasets
import pandas as pd
import numpy as np
from datasets import load_dataset
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

frames = []

print('--- Loading datasets ---')

# Dataset 1: GonzaloA/fake_news (24k, confirmed working April 2026)
try:
    ds = load_dataset('GonzaloA/fake_news', split='train')
    df = ds.to_pandas()
    df['text'] = (df.get('title', pd.Series(['']*len(df))).fillna('') + ' ' + df['text'].fillna('')).str.strip()
    df['label'] = df['label'].astype(int)
    frames.append(df[['text','label']])
    print(f'  [1] GonzaloA/fake_news: {len(df)} rows | labels: {df["label"].value_counts().to_dict()}')
except Exception as e:
    print(f'  [1] GonzaloA/fake_news: FAILED - {e}')

# Dataset 2: Yakoob-Khan
try:
    ds = load_dataset('Yakoob-Khan/Fake-News-Detection', split='train')
    df = ds.to_pandas()
    text_col  = next((c for c in ['text','title','content','statement'] if c in df.columns), df.columns[0])
    label_col = next((c for c in ['label','Label','fake','class'] if c in df.columns), df.columns[-1])
    df['text']  = df[text_col].fillna('').str.strip()
    df['label'] = pd.to_numeric(df[label_col], errors='coerce')
    df = df.dropna(subset=['label'])
    df['label'] = df['label'].astype(int)
    df = df[df['label'].isin([0,1])]
    frames.append(df[['text','label']])
    print(f'  [2] Yakoob-Khan: {len(df)} rows | labels: {df["label"].value_counts().to_dict()}')
except Exception as e:
    print(f'  [2] Yakoob-Khan: FAILED - {e}')

# Dataset 3: saurabhp19
try:
    ds = load_dataset('saurabhp19/fake_news_detection', split='train')
    df = ds.to_pandas()
    text_col  = next((c for c in ['text','title','statement','content'] if c in df.columns), df.columns[0])
    label_col = next((c for c in ['label','Label','fake'] if c in df.columns), df.columns[-1])
    df['text']  = df[text_col].fillna('').str.strip()
    df['label'] = pd.to_numeric(df[label_col], errors='coerce')
    df = df.dropna(subset=['label'])
    df['label'] = df['label'].astype(int)
    df = df[df['label'].isin([0,1])]
    frames.append(df[['text','label']])
    print(f'  [3] saurabhp19: {len(df)} rows | labels: {df["label"].value_counts().to_dict()}')
except Exception as e:
    print(f'  [3] saurabhp19: FAILED - {e}')

# Dataset 4: difraud
try:
    ds = load_dataset('difraud/difraud', split='train')
    df = ds.to_pandas()
    df['text']  = df['text'].fillna('').str.strip()
    df['label'] = df['label'].astype(int)
    df = df.sample(min(15000, len(df)), random_state=42)
    frames.append(df[['text','label']])
    print(f'  [4] difraud: {len(df)} rows | labels: {df["label"].value_counts().to_dict()}')
except Exception as e:
    print(f'  [4] difraud: FAILED - {e}')

print(f'\nFrames loaded: {len(frames)}')
assert len(frames) > 0, 'ERROR: No datasets loaded! Check internet connection.'

# Merge and clean
df_all = pd.concat(frames, ignore_index=True)
df_all = df_all.dropna(subset=['text','label'])
df_all = df_all[df_all['text'].str.len() >= 20]
df_all['text']  = df_all['text'].str[:512]
df_all = df_all.drop_duplicates(subset=['text'])
df_all['label'] = df_all['label'].astype(int)
df_all = df_all[df_all['label'].isin([0,1])]

print(f'\nAfter cleaning: {len(df_all)} rows')
print(f'Label distribution: {df_all["label"].value_counts().to_dict()}')

# Balance
fake   = df_all[df_all['label']==1]
real   = df_all[df_all['label']==0]
target = min(20000, min(len(fake), len(real)))
df_bal = pd.concat([
    fake.sample(target, random_state=42),
    real.sample(target, random_state=42)
]).sample(frac=1, random_state=42).reset_index(drop=True)

train_df, temp = train_test_split(df_bal, test_size=0.15, random_state=42, stratify=df_bal['label'])
val_df, test_df = train_test_split(temp, test_size=0.5, random_state=42, stratify=temp['label'])

# VERIFY
assert len(train_df) > 1000, 'ERROR: Training set too small'
assert train_df['label'].nunique() == 2, 'ERROR: Only one class in training data!'
assert abs(train_df['label'].mean() - 0.5) < 0.05, 'ERROR: Classes not balanced!'
print(f'\nTrain:{len(train_df)} Val:{len(val_df)} Test:{len(test_df)}')
print(f'Train balance: {train_df["label"].value_counts().to_dict()}')
print('CHECK: PASS - data ready')

In [ ]:
# CELL 4 — Tokenize
from transformers import AutoTokenizer
from datasets import Dataset

BASE_MODEL = 'microsoft/deberta-v3-base'
MAX_LEN    = 256

print(f'Loading tokenizer: {BASE_MODEL}')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=MAX_LEN)

def to_ds(df):
    ds = Dataset.from_dict({'text': df['text'].tolist(), 'labels': df['label'].tolist()})
    return ds.map(tokenize, batched=True, batch_size=256, remove_columns=['text'])

print('Tokenizing train...')
train_ds = to_ds(train_df)
print('Tokenizing val...')
val_ds   = to_ds(val_df)
print('Tokenizing test...')
test_ds  = to_ds(test_df)
train_ds.set_format('torch')
val_ds.set_format('torch')
test_ds.set_format('torch')

# VERIFY tokenization
sample = train_ds[0]
assert 'input_ids' in sample, 'ERROR: tokenization failed - no input_ids'
assert 'labels' in sample, 'ERROR: tokenization failed - no labels'
assert len(sample['input_ids']) == MAX_LEN, f'ERROR: wrong token length {len(sample["input_ids"])}'
assert sample['labels'].item() in [0,1], 'ERROR: invalid label value'
print(f'Sample input_ids length: {len(sample["input_ids"])} (expected {MAX_LEN})')
print(f'Sample label: {sample["labels"].item()} (0=REAL, 1=FAKE)')
print('CHECK: PASS - tokenization correct')

In [ ]:
# CELL 5 — Model + Trainer setup
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
import evaluate

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=2,
    id2label={0:'REAL', 1:'FAKE'},
    label2id={'REAL':0, 'FAKE':1},
    ignore_mismatched_sizes=True
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters:    {total_params:,}')
print(f'Trainable parameters:{trainable:,}')

acc_m = evaluate.load('accuracy')
f1_m  = evaluate.load('f1')

def compute_metrics(ep):
    preds = ep[0].argmax(axis=-1)
    return {
        'accuracy': acc_m.compute(predictions=preds, references=ep[1])['accuracy'],
        'f1_macro': f1_m.compute(predictions=preds, references=ep[1], average='macro')['f1']
    }

args = TrainingArguments(
    output_dir='./deberta-factchecker',
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=200,
    lr_scheduler_type='linear',
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    fp16=False,
    bf16=False,
    gradient_accumulation_steps=4,
    max_grad_norm=1.0,
    dataloader_num_workers=2,
    logging_steps=50,
    report_to='none',
    save_total_limit=2,
    seed=42,
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

# VERIFY: run one forward pass to confirm model works before full training
import torch
model.eval()
with torch.no_grad():
    sample_batch = {k: train_ds[0][k].unsqueeze(0).to(device) for k in ['input_ids','attention_mask','token_type_ids'] if k in train_ds[0]}
    out = model(**sample_batch)
    logits = out.logits
    assert logits.shape == (1, 2), f'ERROR: unexpected output shape {logits.shape}'
    pred = logits.argmax(dim=-1).item()
    print(f'Forward pass OK. Sample prediction: {pred} ({"FAKE" if pred==1 else "REAL"}) - random before training')
model.train()
print('CHECK: PASS - model ready for training')

In [ ]:
# CELL 6 — TRAIN
# Expected: Training Loss should DECREASE each epoch (not stay at 0.0)
# Expected: Accuracy should INCREASE above 0.5 after epoch 1
# If Training Loss stays 0.0 or Accuracy stays 0.5 -> something is wrong
print('Starting training (~60-80 min on T4)...')
print('Watch: Training Loss should decrease, Accuracy should increase above 0.5')
result = trainer.train()
print(f'\nDone. Steps:{result.global_step} Loss:{result.training_loss:.4f}')
if result.training_loss < 0.01:
    print('WARNING: Training loss is suspiciously low - check if training actually ran')
else:
    print('CHECK: PASS - training completed with non-zero loss')

In [ ]:
# CELL 7 — Evaluate
from sklearn.metrics import classification_report
from sklearn.metrics import brier_score_loss
import torch.nn.functional as F
import torch, json, datetime, numpy as np, os

res = trainer.evaluate(test_ds)
print(f'Test Accuracy: {res["eval_accuracy"]:.4f}')
print(f'Test F1 Macro: {res["eval_f1_macro"]:.4f}')

out    = trainer.predict(test_ds)
y_pred = out.predictions.argmax(axis=-1)
y_true = out.label_ids
print('\n' + classification_report(y_true, y_pred, target_names=['Real','Fake']))

probs      = F.softmax(torch.tensor(out.predictions.astype(np.float32)), dim=-1).numpy()
fake_probs = np.nan_to_num(probs[:,1], nan=0.5, posinf=1.0, neginf=0.0)
brier      = brier_score_loss(y_true, fake_probs)
print(f'Brier score: {brier:.4f} (lower = better, <0.15 is good)')

# VERIFY results are meaningful
acc = res['eval_accuracy']
f1  = res['eval_f1_macro']
if acc < 0.6:
    print(f'WARNING: Accuracy {acc:.4f} is low - model may not have learned properly')
elif acc < 0.75:
    print(f'CHECK: ACCEPTABLE - Accuracy {acc:.4f}, model learned but could be better')
else:
    print(f'CHECK: PASS - Accuracy {acc:.4f}, F1 {f1:.4f} - good model!')

# Quick sanity test on known examples
from transformers import pipeline
clf = pipeline('text-classification', model=model, tokenizer=tokenizer, device=0 if device=='cuda' else -1)
tests = [
    ('Scientists discover water on Mars in new NASA study', 'REAL'),
    ('SHOCKING: Government putting microchips in vaccines to control population!!!', 'FAKE'),
    ('Stock market closes higher after Fed announcement', 'REAL'),
]
print('\n--- Sanity checks ---')
all_pass = True
for text, expected in tests:
    r = clf(text[:256])[0]
    label = r['label']
    score = r['score']
    status = 'PASS' if label == expected else 'FAIL'
    if status == 'FAIL': all_pass = False
    print(f'  [{status}] {label} ({score:.2f}) | expected {expected} | {text[:60]}...')

if all_pass:
    print('CHECK: PASS - all sanity checks passed')
else:
    print('CHECK: WARN - some sanity checks failed, model may need more training')

# Save version info
os.makedirs('./deberta-factchecker', exist_ok=True)
version = {
    'model': MODEL_NAME, 'base': BASE_MODEL,
    'version': datetime.datetime.utcnow().strftime('%Y%m%d_%H%M'),
    'accuracy': round(float(acc), 4),
    'f1_macro': round(float(f1), 4),
    'brier_score': round(float(brier), 4),
    'train_samples': len(train_df),
    'max_length': MAX_LEN
}
with open('./deberta-factchecker/model_version.json','w') as f:
    json.dump(version, f, indent=2)
print('\n' + json.dumps(version, indent=2))

In [ ]:
# CELL 8 — Push to HuggingFace + download artifacts
if HF_TOKEN:
    print(f'Pushing to {MODEL_NAME}...')
    trainer.push_to_hub(MODEL_NAME)
    tokenizer.push_to_hub(MODEL_NAME)
    print(f'Live at: https://huggingface.co/{MODEL_NAME}')
    print('CHECK: PASS - model pushed to HuggingFace')
else:
    trainer.save_model('./deberta-factchecker')
    tokenizer.save_pretrained('./deberta-factchecker')
    print('Saved locally (no HF token)')

from google.colab import files
files.download('./deberta-factchecker/model_version.json')
print('\nDownloaded model_version.json - drop into backend/data/')
print(f'Then add Render env var: DEBERTA_MODEL={MODEL_NAME}')